# Workflow Parameter Convergence

Run a convergence study for **any parameter** in a single-unit workflow and return the converged value for reuse in another notebook.

The parameter is injected into the unit's input template via regex substitution, so any plaintext assignment in the input (e.g. `degauss = 0.005`) can be turned into a convergence variable.

<h2 style="color:green">Usage</h2>

1. Set material, workflow, convergence, and compute parameters in cell 1.2. below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to review the convergence series and the converged parameter value.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the material, workflow, convergence loop, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Build the convergence workflow: load the workflow from Standata, inject the convergence parameter via regex into the unit's input template, and add the convergence loop.
1. Configure compute: get the list of clusters and create compute configuration with the selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from the saved material, convergence-enabled workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: reconstruct the convergence series from `job.scopeTrack` and report the converged parameter value.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("api_examples")

### 1.2. Set parameters and configurations for the convergence job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 1. Organization / account selection
ORGANIZATION_NAME = None

# 2. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 3. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "total_energy.json"

# 4. Convergence parameter
# Common parameters to converge:
# - ecutwfc: wavefunction cutoff (most important) — start ~20 Ry, increment 5-10 Ry
# - ecutrho: charge density cutoff — start 4×ecutwfc, increment 20-40 Ry
# - degauss: smearing width (metals) — start 0.001 Ry, increment 0.002 Ry
# Advanced (workflow-specific):
# - ecutfock: Fock exchange cutoff (HSE) — start ~80 Ry, increment 20 Ry
# - tr2_ph: phonon self-consistency threshold — start 1e-12, decrease by 10×
# - num_band: number of bands (GW) — start 8, increment 2-4
# - ecut_corr: correlation cutoff (GW) — start 5 Ry, increment 1 Ry

PARAMETER_NAME = "ecutwfc" # adjust to desired parameter in the input template
PARAMETER_INITIAL = 20
PARAMETER_INCREMENT = 5
INPUT_NAME = None  # if provided, targets a specific input file by name (e.g. "pw_scf.in", "INCAR") only

# 5. Convergence result
# Common results to monitor:
# - total_energy: standard baseline
# - total_force: for relaxations/phonons
# - fermi_energy: for metals, work function calculations
# - pressure: for equation of state, elastic constants
RESULT_NAME = "total_energy" # adjust to desired result in the output template
RESULT_INITIAL = 0
TOLERANCE = 1e-4 # for comparison

# 6. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 7. Job naming
RUN_LABEL = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and store the credentials in the current environment.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
PROJECT_ID = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({PROJECT_ID})")

## 3. Load and save the material
### 3.1. Load material from local uploads or Standata

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)
print(f"Saved material ID: {saved_material.id}")

## 4. Build the convergence workflow
### 4.1. Load the workflow

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = f"Convergence - {RUN_LABEL}"

visualize_workflow(workflow)

### 4.2. Inject convergence parameter and add convergence loop

The regex `PARAM_NAME = <numeric_value>` is auto-generated from `PARAM_NAME`, locating the existing assignment in the input template and replacing it with a runtime scope variable. The convergence loop then varies that variable each iteration.

In [ ]:
convergence_subworkflow = workflow.subworkflows[0]
convergence_subworkflow.add_template_parameter_convergence(
    parameter_name=PARAMETER_NAME,
    parameter_initial=PARAMETER_INITIAL,
    parameter_increment=PARAMETER_INCREMENT,
    result_name=RESULT_NAME,
    result_initial=RESULT_INITIAL,
    input_name=INPUT_NAME,
    tolerance=TOLERANCE,
)

print(f"Convergence parameter: {convergence_subworkflow.convergence_parameter}")
print(f"Convergence result: {convergence_subworkflow.convergence_result}")
visualize_workflow(workflow)

## 5. Configure compute
### 5.1. Configure compute

In [ ]:
from mat3ra.ide.compute import Compute

clusters = client.clusters.list()
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration
### 6.1. Create the job with the embedded convergence workflow

In [ ]:
from mat3ra.notebooks_utils.api.job import create_job
from utils.generic import dict_to_namespace
from mat3ra.notebooks_utils.ui import display_JSON

job_name = f"{workflow.name} {saved_material.formula}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=PROJECT_ID,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace(job_response)
job_id = job._id
print(f"Job created: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"Submitted job: {job_id}")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async
await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results
### 8.1. Fetch the finished job and reconstruct the convergence series

In [ ]:
from mat3ra.notebooks_utils.api.job import get_convergence_series
from mat3ra.notebooks_utils.ipython.plot._matplotlib import plot_series

series = get_convergence_series(client, job_id)

print("Convergence series:")
for item in series:
    print(item)

plot_series(
    series=series,
    x_key="parameter",
    y_key="y",
    xlabel=f"{PARAMETER_NAME}",
    ylabel=f"{RESULT_NAME}",
    title=f"Convergence: {PARAMETER_NAME}",
)

### 8.2. Report the converged parameter value
Use `converged_value` in another notebook when setting the parameter for the production calculation.

In [ ]:
converged_value = series[-1]["parameter"]
print(f"Convergence parameter: {PARAMETER_NAME}")
print(f"Converged value: {converged_value} ")